# Supply Chain & Port Operations Analytics During Strait of Hormuz Disruption



## Engineer 1: ETL & Data Quality Engineer


In [84]:
!pip install  pyspark findspark 

### Task 1: Data Ingestion & Validation


Created the spark Session

In [85]:
from pyspark.sql import SparkSession #type:ignore
from pyspark.sql.functions import *   #type:ignore


spark=SparkSession.builder\
    .appName("Supply_Chain_ETL_Project")\
        .getOrCreate()

### Google Drive Mount

In [86]:
from google.colab import drive # type:ignore
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Load both datasets into Spark DataFrames

In [87]:
print("Loding the Dataset_1 (Supply_Chain) from Drive")
df_1=spark.read.csv("/content/drive/MyDrive/supply_chain_hormuz_crisis_700.csv", header=True,inferSchema=True)

print("Loding the Dataset_2(Port_Operations) from Drive")
df_2=spark.read.csv("/content/drive/MyDrive/port_operations_master.csv", header=True,inferSchema=True)


Loding the Dataset_1 (Supply_Chain) from Drive
Loding the Dataset_2(Port_Operations) from Drive


In [88]:
print("Schema_1")
df_1.printSchema()


Schema_1
root
 |-- Shipment_ID: string (nullable = true)
 |-- Supplier_ID: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Product_Type: string (nullable = true)
 |-- Monthly_Demand_Tons: integer (nullable = true)
 |-- Shipment_Volume_Tons: integer (nullable = true)
 |-- Route_Risk_Score: double (nullable = true)
 |-- Historical_Delay_Days: integer (nullable = true)
 |-- Fuel_Price_USD: double (nullable = true)
 |-- Political_Risk_Index: double (nullable = true)
 |-- Port_Congestion_Index: double (nullable = true)
 |-- Inventory_Days: integer (nullable = true)
 |-- Supplier_Reliability: double (nullable = true)
 |-- Alternative_Supplier_Count: integer (nullable = true)
 |-- Transit_Time_Days: integer (nullable = true)
 |-- Delay_Probability: double (nullable = true)
 |-- Current_Delay_Days: integer (nullable = true)
 |-- Freight_Cost_USD: double (nullable = true)
 |-- Revenue_Impact_USD: double (nullable = true)
 |-- Disruption_Event: integer (nullable = true)



In [89]:
print("Schema_2")
df_2.printSchema()

Schema_2
root
 |-- Port_ID: string (nullable = true)
 |-- Port_Name: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Port_Type: string (nullable = true)
 |-- Port_Capacity_Tons: integer (nullable = true)
 |-- Avg_Vessel_Wait_Time_Hours: double (nullable = true)
 |-- Port_Utilization_Percent: double (nullable = true)
 |-- Active_Vessels: integer (nullable = true)
 |-- Weather_Disruption_Score: double (nullable = true)
 |-- Labor_Availability_Percent: double (nullable = true)
 |-- Monthly_Port_Revenue_USD: double (nullable = true)
 |-- Operational_Status: string (nullable = true)
 |-- Port_Manager: string (nullable = true)
 |-- Last_Inspection_Date: date (nullable = true)



### Counting Number Of Records

In [90]:
print(f"Record Count Dataset_1: {df_1.count()}") #type:ignore
print(f"Record Count Dataset_2: {df_2.count()}") #type:ignore

Record Count Dataset_1: 700
Record Count Dataset_2: 300


### Show() The Dataset

In [91]:
df_1.show()

+-----------+-----------+------------+--------------+-------------------+--------------------+----------------+---------------------+--------------+--------------------+---------------------+--------------+--------------------+--------------------------+-----------------+-----------------+------------------+----------------+------------------+----------------+
|Shipment_ID|Supplier_ID|     Country|  Product_Type|Monthly_Demand_Tons|Shipment_Volume_Tons|Route_Risk_Score|Historical_Delay_Days|Fuel_Price_USD|Political_Risk_Index|Port_Congestion_Index|Inventory_Days|Supplier_Reliability|Alternative_Supplier_Count|Transit_Time_Days|Delay_Probability|Current_Delay_Days|Freight_Cost_USD|Revenue_Impact_USD|Disruption_Event|
+-----------+-----------+------------+--------------+-------------------+--------------------+----------------+---------------------+--------------+--------------------+---------------------+--------------+--------------------+--------------------------+-----------------+--

In [92]:
df_2.show()

+-------+--------------+------------+--------------+------------------+--------------------------+------------------------+--------------+------------------------+--------------------------+------------------------+------------------+------------+--------------------+
|Port_ID|     Port_Name|     Country|     Port_Type|Port_Capacity_Tons|Avg_Vessel_Wait_Time_Hours|Port_Utilization_Percent|Active_Vessels|Weather_Disruption_Score|Labor_Availability_Percent|Monthly_Port_Revenue_USD|Operational_Status|Port_Manager|Last_Inspection_Date|
+-------+--------------+------------+--------------+------------------+--------------------------+------------------------+--------------+------------------------+--------------------------+------------------------+------------------+------------+--------------------+
|PORT007|Singapore Port|     Germany|  LNG Terminal|            394138|                     43.17|                   53.41|           130|                    1.52|                     72.37|   

### Null Count

In [93]:
df_1_NullCounts=df_1.select([count(when(col(c).isNull(),c)).alias(c)for c in df_1.columns]).show() #type:ignore

df_2_NullCount=df_2.select([count(when(col(c).isNull(),c)).alias(c)for c in df_2.columns]).show() #type:ignore

+-----------+-----------+-------+------------+-------------------+--------------------+----------------+---------------------+--------------+--------------------+---------------------+--------------+--------------------+--------------------------+-----------------+-----------------+------------------+----------------+------------------+----------------+
|Shipment_ID|Supplier_ID|Country|Product_Type|Monthly_Demand_Tons|Shipment_Volume_Tons|Route_Risk_Score|Historical_Delay_Days|Fuel_Price_USD|Political_Risk_Index|Port_Congestion_Index|Inventory_Days|Supplier_Reliability|Alternative_Supplier_Count|Transit_Time_Days|Delay_Probability|Current_Delay_Days|Freight_Cost_USD|Revenue_Impact_USD|Disruption_Event|
+-----------+-----------+-------+------------+-------------------+--------------------+----------------+---------------------+--------------+--------------------+---------------------+--------------+--------------------+--------------------------+-----------------+-----------------+-----

### Duplicate records

In [94]:
duplicate_1=df_1.count()-df_1.dropDuplicates().count()
print(f"Duplicate Records For Dataset_1 : {duplicate_1}")

duplicate_2=df_2.count()-df_2.dropDuplicates().count()
print(f"Duplicate Records For Dataset_2 : {duplicate_2}")


Duplicate Records For Dataset_1 : 0
Duplicate Records For Dataset_2 : 0


### Invalid Port_ID values

In [95]:
valid_pattern = r"^PORT\d{3}$"
invalid_port_df=df_2.filter(col('Port_ID').isNull() | (~col('Port_ID').rlike(valid_pattern))) # type:ignore
invalid_port_count=invalid_port_df.count()
invalid_port_count

0

### Generate a Data Quality Summary Report

In [96]:
print("--- DATASET 1 SUMMARY ---")
print(f"Total Records: {df_1.count()}")
print(f"Null Counts:   {df_1_NullCounts}")
print(f"Duplicates:    {duplicate_1}")


print("--- DATASET 2 SUMMARY ---")
print(f"Total Records: {df_2.count()}")
print(f"Null Counts:   {df_2_NullCount}")
print(f"Duplicates:    {duplicate_2}")
print(f"Invalid Ports: {invalid_port_count}")

--- DATASET 1 SUMMARY ---
Total Records: 700
Null Counts:   None
Duplicates:    0
--- DATASET 2 SUMMARY ---
Total Records: 300
Null Counts:   None
Duplicates:    0
Invalid Ports: 0


## Task 2: Data Transformation Pipeline

In [97]:
df_1.show()

+-----------+-----------+------------+--------------+-------------------+--------------------+----------------+---------------------+--------------+--------------------+---------------------+--------------+--------------------+--------------------------+-----------------+-----------------+------------------+----------------+------------------+----------------+
|Shipment_ID|Supplier_ID|     Country|  Product_Type|Monthly_Demand_Tons|Shipment_Volume_Tons|Route_Risk_Score|Historical_Delay_Days|Fuel_Price_USD|Political_Risk_Index|Port_Congestion_Index|Inventory_Days|Supplier_Reliability|Alternative_Supplier_Count|Transit_Time_Days|Delay_Probability|Current_Delay_Days|Freight_Cost_USD|Revenue_Impact_USD|Disruption_Event|
+-----------+-----------+------------+--------------+-------------------+--------------------+----------------+---------------------+--------------+--------------------+---------------------+--------------+--------------------+--------------------------+-----------------+--

## Freight_Cost_Per_Ton


In [98]:
df_1_transformed = df_1.withColumn("Freight_Cost_Per_Ton", round(col('Freight_Cost_USD') / col('Shipment_Volume_Tons'), 2))\
        .withColumn("Delay_Category",
                    when(col("Current_Delay_Days") <= 5, "Low") 
        .when((col("Current_Delay_Days") >= 6) & (col("Current_Delay_Days") <= 15), "Medium") 
        .otherwise("High"))\
                .withColumn("Inventory_Risk",
                            when(col("Current_Delay_Days") > col("Inventory_Days"), "Yes") 
                            .otherwise("No")
    )
                
df_1_transformed.show()      

+-----------+-----------+------------+--------------+-------------------+--------------------+----------------+---------------------+--------------+--------------------+---------------------+--------------+--------------------+--------------------------+-----------------+-----------------+------------------+----------------+------------------+----------------+--------------------+--------------+--------------+
|Shipment_ID|Supplier_ID|     Country|  Product_Type|Monthly_Demand_Tons|Shipment_Volume_Tons|Route_Risk_Score|Historical_Delay_Days|Fuel_Price_USD|Political_Risk_Index|Port_Congestion_Index|Inventory_Days|Supplier_Reliability|Alternative_Supplier_Count|Transit_Time_Days|Delay_Probability|Current_Delay_Days|Freight_Cost_USD|Revenue_Impact_USD|Disruption_Event|Freight_Cost_Per_Ton|Delay_Category|Inventory_Risk|
+-----------+-----------+------------+--------------+-------------------+--------------------+----------------+---------------------+--------------+--------------------+---

## Port_Load_Category

In [99]:
df_2.show()

+-------+--------------+------------+--------------+------------------+--------------------------+------------------------+--------------+------------------------+--------------------------+------------------------+------------------+------------+--------------------+
|Port_ID|     Port_Name|     Country|     Port_Type|Port_Capacity_Tons|Avg_Vessel_Wait_Time_Hours|Port_Utilization_Percent|Active_Vessels|Weather_Disruption_Score|Labor_Availability_Percent|Monthly_Port_Revenue_USD|Operational_Status|Port_Manager|Last_Inspection_Date|
+-------+--------------+------------+--------------+------------------+--------------------------+------------------------+--------------+------------------------+--------------------------+------------------------+------------------+------------+--------------------+
|PORT007|Singapore Port|     Germany|  LNG Terminal|            394138|                     43.17|                   53.41|           130|                    1.52|                     72.37|   

In [100]:
df_2_transformed = df_2.withColumn("Port_Load_Category",
        when(col("Port_Utilization_Percent") < 60, "Low") 
         .when((col("Port_Utilization_Percent") >= 60) & (col("Port_Utilization_Percent") <= 80), "Medium")
         .otherwise("High")
    )

df_2_transformed.show()

+-------+--------------+------------+--------------+------------------+--------------------------+------------------------+--------------+------------------------+--------------------------+------------------------+------------------+------------+--------------------+------------------+
|Port_ID|     Port_Name|     Country|     Port_Type|Port_Capacity_Tons|Avg_Vessel_Wait_Time_Hours|Port_Utilization_Percent|Active_Vessels|Weather_Disruption_Score|Labor_Availability_Percent|Monthly_Port_Revenue_USD|Operational_Status|Port_Manager|Last_Inspection_Date|Port_Load_Category|
+-------+--------------+------------+--------------+------------------+--------------------------+------------------------+--------------+------------------------+--------------------------+------------------------+------------------+------------+--------------------+------------------+
|PORT007|Singapore Port|     Germany|  LNG Terminal|            394138|                     43.17|                   53.41|           13

## Task 3: Data Lake Layer Creation

### Bronze Layer

In [101]:
import os

path = "/content/drive/MyDrive/Supply_Chain_ETL_Project/data_lake_layer"

for layer in ['bronze', 'silver', 'gold']:
    folder = f"{path}/{layer}"
    os.makedirs(folder, exist_ok=True)
    
df_1.write.mode("overwrite").parquet(f"{path}/bronze/raw_1_data.parquet")
df_2.write.mode("overwrite").parquet(f"{path}/bronze/raw_2_data.parquet")

print("Saved to Bronze layer in Google Drive.")
    

Saved to Bronze layer in Google Drive.


In [102]:
df_1_transformed.write.mode("overwrite").parquet(f"{path}/silver/transformed_data1.parquet")
df_2_transformed.write.mode("overwrite").parquet(f"{path}/silver/transformed_data2.parquet")

print("Transformed Data Saved to Silver layer in Google Drive.")

Transformed Data Saved to Silver layer in Google Drive.
